# Theme 3: Agent Comparison & Temporal Benchmarking

**RQ6:** Which AI agent is *best at bug fixing*?  
**RQ8:** How does performance compare *before and after the AIDev cutoff*?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)  
AIDev period: Dec 2024 – Jul 2025 | Post-AIDev: Aug 2025 – Feb 2026

In [ ]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

In [ ]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [ ]:
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs()
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()
commits    = load_commits()
details    = load_commit_details()
rev_stats  = build_revision_stats(df, commits, details)
agent_rev  = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
human_rev  = rev_stats[~rev_stats['is_agent']].copy()
print('All data loaded.')

## RQ6: Which AI agent is best at bug fixing?

Metrics: merge rate, time to merge, revision rate, revision effort.  
Statistical tests vs Human baseline: chi-square (merge rate), Mann-Whitney U (continuous).

In [ ]:
# Overall merge rate — all agents vs human
h_m, h_t, h_r = merge_rate(human_df)
print("{:<15} {:>8} {:>8} {:>8}  {}".format('Group', 'Merged', 'Total', 'Rate%', 'vs Human'))
print('-' * 55)
print("{:<15} {:>8,} {:>8,} {:>7.1f}%".format('Human', h_m, h_t, h_r))
for agent in AGENTS:
    sub = agents_df[agents_df['agent'] == agent]
    a_m, a_t, a_r = merge_rate(sub)
    chi2, p = chi_square(a_m, a_t, h_m, h_t)
    print("{:<15} {:>8,} {:>8,} {:>7.1f}%  chi2={:.1f} p={:.4f} {}".format(
        agent, a_m, a_t, a_r, chi2, p, sig_label(p)))

In [ ]:
# Figure: merge rate bar chart
groups = AGENTS + ['Human']
rates  = []
for g in groups:
    if g == 'Human':
        _, _, r = merge_rate(human_df)
    else:
        _, _, r = merge_rate(agents_df[agents_df['agent'] == g])
    rates.append(r)

colors = [AGENT_COLORS[g] for g in groups]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(groups, rates, color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10)
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ6: Bug-Fix Merge Rate by Agent vs Human')
fig.tight_layout()
save_fig(fig, 'rq6_merge_rate', THEME3_DIR)

In [ ]:
# Time to merge — median + Mann-Whitney vs Human
h_ttm = human_df.loc[human_df['is_merged'], 'hours_to_merge']
print("{:<15} {:>16}  Mann-Whitney vs Human".format('Group', 'Median TTM (h)'))
print('-' * 55)
print("{:<15} {:>16.2f}".format('Human', h_ttm.median()))
for agent in AGENTS:
    sub = agents_df[(agents_df['agent'] == agent) & agents_df['is_merged']]
    u, p = mann_whitney(sub['hours_to_merge'], h_ttm)
    print("{:<15} {:>16.2f}  U={:.0f} p={:.4f} {}".format(
        agent, sub['hours_to_merge'].median(), u, p, sig_label(p)))

In [ ]:
# Figure: time to merge — boxplot (capped at 48h for readability)
CAP = 48
plot_data = []
labels    = []
for agent in AGENTS:
    sub = agents_df[(agents_df['agent'] == agent) & agents_df['is_merged']]
    plot_data.append(sub['hours_to_merge'].clip(upper=CAP).dropna())
    labels.append(agent)
plot_data.append(h_ttm.clip(upper=CAP).dropna())
labels.append('Human')

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(plot_data, labels=labels, patch_artist=True, notch=False)
for patch, label in zip(bp['boxes'], labels):
    patch.set_facecolor(AGENT_COLORS[label])
    patch.set_alpha(0.75)
ax.set_ylabel(f'Hours to Merge (capped at {CAP}h)')
ax.set_title('RQ6: Time to Merge Distribution per Agent vs Human')
fig.tight_layout()
save_fig(fig, 'rq6_time_to_merge', THEME3_DIR)

In [ ]:
# Revision burden summary: % revised + median revision lines
h_rev_pct = (human_rev['num_commits'] > 1).mean() * 100
h_rev_add = human_rev.loc[human_rev['num_commits'] > 1, 'rev_lines_added'].median()
h_rev_del = human_rev.loc[human_rev['num_commits'] > 1, 'rev_lines_deleted'].median()

print("{:<15} {:>10} {:>16} {:>16}".format(
    'Group', '% Revised', 'Med RevLines+', 'Med RevLines-'))
print('-' * 60)
print("{:<15} {:>9.1f}% {:>16.0f} {:>16.0f}".format(
    'Human', h_rev_pct, h_rev_add, h_rev_del))
for agent in AGENTS:
    sub     = agent_rev[agent_rev['agent'] == agent]
    pct     = (sub['num_commits'] > 1).mean() * 100
    revised = sub[sub['num_commits'] > 1]
    add     = revised['rev_lines_added'].median()
    dlt     = revised['rev_lines_deleted'].median()
    u_add, p_add = mann_whitney(
        revised['rev_lines_added'],
        human_rev.loc[human_rev['num_commits'] > 1, 'rev_lines_added']
    )
    print("{:<15} {:>9.1f}% {:>16.0f} {:>16.0f}  p(rev_lines)={:.4f} {}".format(
        agent, pct, add, dlt, p_add, sig_label(p_add)))

In [ ]:
# Figure: revision metrics bar chart (4 subplots)
metrics   = ['Merge Rate %', 'Median TTM (h)', '% Revised', 'Med Rev Lines+']
groups    = AGENTS + ['Human']
all_vals  = {g: [] for g in groups}

for g in groups:
    sub = agents_df[agents_df['agent'] == g] if g != 'Human' else human_df
    _, _, mr = merge_rate(sub)
    ttm = sub.loc[sub['is_merged'], 'hours_to_merge'].median()
    rev_sub = agent_rev[agent_rev['agent'] == g] if g != 'Human' else human_rev
    rev_pct = (rev_sub['num_commits'] > 1).mean() * 100
    rev_add = rev_sub.loc[rev_sub['num_commits'] > 1, 'rev_lines_added'].median()
    all_vals[g] = [mr, ttm, rev_pct, rev_add]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for i, (ax, metric) in enumerate(zip(axes, metrics)):
    vals   = [all_vals[g][i] for g in groups]
    colors = [AGENT_COLORS[g] for g in groups]
    ax.bar(groups, vals, color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=10)
    ax.set_xticklabels(groups, rotation=30, ha='right', fontsize=8)
fig.suptitle('RQ6: Agent Comparison — Key Bug-Fix Metrics', fontsize=12)
fig.tight_layout()
save_fig(fig, 'rq6_summary', THEME3_DIR)

## RQ8: AIDev period vs Post-AIDev period

Does agent bug-fixing performance change after the AIDev coverage ends?

- **AIDev period:** Dec 2024 – Jul 2025  
- **Post-AIDev period:** Aug 2025 – Feb 2026

In [ ]:
# Merge rate per agent — AIDev vs Post-AIDev
PERIODS = ['AIDev (Dec24-Jul25)', 'Post-AIDev (Aug25-Feb26)']
rows_rq8 = []
for agent in AGENTS:
    sub = agents_df[agents_df['agent'] == agent]
    for period in PERIODS:
        p_sub = sub[sub['period'] == period]
        m, t, r = merge_rate(p_sub)
        rows_rq8.append({'Agent': agent, 'Period': period,
                         'Merged': m, 'Total': t, 'Rate': r})

rq8_df = pd.DataFrame(rows_rq8)
pivot  = rq8_df.pivot(index='Agent', columns='Period', values='Rate').round(1)
pivot.columns.name = None
pivot['Change (pp)'] = (
    pivot['Post-AIDev (Aug25-Feb26)'] - pivot['AIDev (Dec24-Jul25)']
).round(1)
print('Merge rate (%) — AIDev vs Post-AIDev:')
print(pivot.to_string())

In [ ]:
# Statistical test: is the period change significant per agent?
print("{:<15} {:>10} {:>10} {:>8} {:>10} {}".format(
    'Agent', 'AIDev %', 'Post %', 'chi2', 'p', 'sig'))
print('-' * 65)
for agent in AGENTS:
    sub  = agents_df[agents_df['agent'] == agent]
    ai   = sub[sub['period'] == 'AIDev (Dec24-Jul25)']
    post = sub[sub['period'] == 'Post-AIDev (Aug25-Feb26)']
    if len(ai) < 5 or len(post) < 5:
        continue
    ai_m, ai_t, ai_r   = merge_rate(ai)
    po_m, po_t, po_r   = merge_rate(post)
    chi2, p            = chi_square(ai_m, ai_t, po_m, po_t)
    print("{:<15} {:>9.1f}% {:>9.1f}% {:>8.2f} {:>10.4f} {}".format(
        agent, ai_r, po_r, chi2, p, sig_label(p)))

In [ ]:
# Figure: grouped bar — AIDev vs Post-AIDev per agent
x     = list(range(len(AGENTS)))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
for i, period in enumerate(PERIODS):
    vals   = [rq8_df[(rq8_df['Agent'] == a) & (rq8_df['Period'] == period)]['Rate'].values[0]
              for a in AGENTS]
    offset = (i - 0.5) * width
    bars   = ax.bar([xi + offset for xi in x], vals, width,
                    label=period, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(AGENTS)
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ8: Bug-Fix Acceptance Rate — AIDev vs Post-AIDev Period')
ax.legend()
fig.tight_layout()
save_fig(fig, 'rq8_aidev_vs_post_aidev', THEME3_DIR)

In [ ]:
# Time to merge: AIDev vs Post-AIDev per agent
print("{:<15} {:>14} {:>14}  Mann-Whitney".format(
    'Agent', 'AIDev TTM (h)', 'Post TTM (h)'))
print('-' * 60)
for agent in AGENTS:
    sub  = agents_df[(agents_df['agent'] == agent) & agents_df['is_merged']]
    ai   = sub[sub['period'] == 'AIDev (Dec24-Jul25)']['hours_to_merge']
    post = sub[sub['period'] == 'Post-AIDev (Aug25-Feb26)']['hours_to_merge']
    if len(ai) < 5 or len(post) < 5:
        continue
    u, p = mann_whitney(ai, post)
    print("{:<15} {:>14.2f} {:>14.2f}  U={:.0f} p={:.4f} {}".format(
        agent, ai.median(), post.median(), u, p, sig_label(p)))